# Ten-fold cross-validation

All images of the dataset are pooled and split into ten folds stratified by
the number of annotated berries. In round k, fold k is the test set, folds
k+1 and k+2 (circularly) are the validation set, used only for early stopping
and for choosing best.pt, and the other folds are the training set. Run folds
0 to 9, one per session, and run the summary cell when all folds are done.

## Setup

Select a GPU runtime (**Runtime > Change runtime type**). Outputs are written
to Google Drive, so that a run interrupted by the end of the session can be
resumed. The Roboflow API key is read from the Colab secret
`ROBOFLOW_API_KEY` (key icon in the left panel).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q ultralytics==8.4.157 roboflow

In [ ]:
import csv
import json
import os
from collections import Counter
from pathlib import Path
from types import SimpleNamespace
import hashlib
import random
import re
import shutil
import statistics
from collections import defaultdict

import yaml

from google.colab import drive, userdata

drive.mount("/content/drive")
OUTPUT = "/content/drive/MyDrive/grape-segmentation"
os.makedirs(OUTPUT, exist_ok=True)
os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")

## Configuration

Hyperparameters kept identical across every run.

In [ ]:
IMAGE_SIZE = 1024

MAX_DETECTIONS = 300

# Hyperparameters kept identical across every run. Most are Ultralytics
# defaults, written out so that a different library version cannot change
# them silently.
TRAINING = dict(
    imgsz=IMAGE_SIZE,
    optimizer="auto",
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    box=7.5,
    cls=0.5,
    dfl=1.5,
    # Masks of overlapping instances are kept separate; with the default
    # (True) a berry mask overwrites the mask of the bunch that contains it.
    overlap_mask=False,
    mask_ratio=4,
    close_mosaic=10,
    amp=True,
    max_det=MAX_DETECTIONS,
    cache="disk",
    plots=True,
    val=True,
)

ONLINE_AUGMENTATION = dict(
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    erasing=0.4,
    degrees=0.0,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    mixup=0.0,
    cutmix=0.0,
    copy_paste=0.0,
)

NO_ONLINE_AUGMENTATION = {name: 0.0 for name in ONLINE_AUGMENTATION}

NO_ONLINE_AUGMENTATION["close_mosaic"] = 0

# Epoch limit and early-stopping patience. Runs on the dataset with offline
# augmentation see ten copies of each image per epoch, so they use fewer epochs.
SCHEDULE_OFFLINE = dict(epochs=150, patience=40)

SCHEDULE_NO_OFFLINE = dict(epochs=600, patience=80)

SCHEDULE_VARIABILITY = dict(epochs=600, patience=120)

## Dataset

The Roboflow export stores part of the instances as bounding boxes. They are
converted into four-vertex polygons, because Ultralytics discards label files
that mix boxes and polygons in segmentation training. Version 14 is the export
without offline augmentation.

In [ ]:
SPLITS = ("train", "valid", "test")


def download_dataset(workspace, project, version, location):
    """Download a Roboflow dataset version in YOLO segmentation format.

    The API key is read from the ROBOFLOW_API_KEY environment variable.
    Returns the path of the generated data.yaml.
    """
    from roboflow import Roboflow

    api_key = os.environ.get("ROBOFLOW_API_KEY")
    if not api_key:
        raise RuntimeError("set the ROBOFLOW_API_KEY environment variable")
    rf = Roboflow(api_key=api_key)
    rf.workspace(workspace).project(project).version(version).download("yolov11", location=str(location))
    return Path(location) / "data.yaml"


def _parse(line):
    parts = line.split()
    if not parts:
        return None, None, None
    values = [float(v) for v in parts[1:]]
    if len(values) == 4:
        return parts[0], values, "box"
    if len(values) >= 6 and len(values) % 2 == 0:
        return parts[0], values, "polygon"
    return parts[0], values, None


def _box_to_polygon(values):
    xc, yc, w, h = values
    x1, x2 = xc - w / 2, xc + w / 2
    y1, y2 = yc - h / 2, yc + h / 2
    return [min(max(p, 0.0), 1.0) for p in (x1, y1, x2, y1, x2, y2, x1, y2)]


def convert_boxes_to_polygons(root):
    """Rewrite bounding-box label lines as four-vertex polygons.

    The Roboflow export stores part of the instances as boxes. A label file
    that mixes boxes and polygons is discarded by Ultralytics in segmentation
    training, so every box is converted to the polygon of its corners.
    Returns the number of converted lines per class.
    """
    root = Path(root)
    converted = Counter()
    for split in SPLITS:
        folder = root / split / "labels"
        if not folder.is_dir():
            continue
        for path in folder.glob("*.txt"):
            lines, changed = [], False
            for line in path.read_text().splitlines():
                cls, values, kind = _parse(line)
                if kind == "box":
                    points = _box_to_polygon(values)
                    lines.append(cls + " " + " ".join(f"{p:.6f}" for p in points))
                    converted[cls] += 1
                    changed = True
                elif line.strip():
                    lines.append(line.strip())
            if changed:
                path.write_text("\n".join(lines) + "\n")
    # Stale label caches would keep the discarded files out of training.
    for cache in root.rglob("*.cache"):
        cache.unlink()
    return converted

In [ ]:
VERSION = 14  # @param {type:"integer"}

data_yaml = download_dataset("hugo-leal-cn7iw", "grapes-mic3h", VERSION, "/content/dataset")
converted = convert_boxes_to_polygons("/content/dataset")
print(f"data.yaml: {data_yaml}")
print(f"box labels converted to polygons: {dict(converted) or 'none'}")

## Metrics

Precision, recall, mAP@50 and mAP@50-95 reported by Ultralytics for boxes and
masks, and F1 computed from precision and recall.

In [ ]:
METRICS = ("P", "R", "mAP50", "mAP50_95")


def f1(p, r):
    return 2 * p * r / (p + r) if p + r else 0.0


def metrics_dict(results, names):
    """Precision, recall, F1, mAP50 and mAP50-95 for boxes and masks.

    Per-class F1 is computed from the precision and recall reported by the
    library; F1 over both classes is the mean of the per-class values, the
    same criterion the library applies to precision and recall.
    """
    def block(m, i=None):
        if i is None:
            values = dict(P=m.mp, R=m.mr, mAP50=m.map50, mAP50_95=m.map)
        else:
            values = dict(P=m.p[i], R=m.r[i], mAP50=m.ap50[i], mAP50_95=m.ap[i])
        return {k: round(float(v), 5) for k, v in values.items()}

    out = {}
    for i, c in enumerate(results.box.ap_class_index):
        out[names[int(c)]] = {"box": block(results.box, i), "mask": block(results.seg, i)}
    out["all"] = {"box": block(results.box), "mask": block(results.seg)}
    for kind in ("box", "mask"):
        per_class = []
        for name, values in out.items():
            values[kind]["F1"] = round(f1(values[kind]["P"], values[kind]["R"]), 5)
            if name != "all":
                per_class.append(values[kind]["F1"])
        out["all"][kind]["F1"] = round(sum(per_class) / len(per_class), 5)
    return out


def table(metrics):
    """Plain-text table of a metrics_dict result."""
    header = f"{'class':<18}" + "".join(f"{k + '(' + t[0].upper() + ')':>12}" for t in ("box", "mask") for k in METRICS + ("F1",))
    rows = [header]
    for name in sorted(metrics, key=lambda n: (n != "all", n)):
        rows.append(f"{name:<18}" + "".join(f"{metrics[name][t][k]:>12.4f}" for t in ("box", "mask") for k in METRICS + ("F1",)))
    return "\n".join(rows)


def best_epoch(run_dir):
    """Epoch of best.pt, using the fitness Ultralytics applies to segmentation."""
    with open(Path(run_dir) / "results.csv", newline="") as f:
        rows = [{k.strip(): v for k, v in row.items()} for row in csv.DictReader(f)]

    def fitness(row):
        return sum(0.1 * float(row[f"metrics/mAP50({t})"]) + 0.9 * float(row[f"metrics/mAP50-95({t})"])
                   for t in ("B", "M"))

    best = max(rows, key=fitness)
    return int(float(best["epoch"])), len(rows)


def save_json(record, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
    return path

## Folds

The split is deterministic and its SHA-256 fingerprint is checked against the
fingerprint of the split used in the study.

In [ ]:
K = 10

SPLIT_SEED = 2026

BERRY_CLASS = 1  # names: ['bunch of grapes', 'grape']

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def group_key(stem):
    """'00094_jpg.rf.01f6...' -> '00094', so that copies of a photo stay together."""
    base = stem.split(".rf.")[0]
    return re.sub(r"_(jpg|jpeg|png|bmp|webp)$", "", base, flags=re.I)


def collect_groups(root):
    groups = defaultdict(list)
    for split in SPLITS:
        folder = root / split / "images"
        if not folder.is_dir():
            continue
        for image in sorted(folder.iterdir()):
            if image.suffix.lower() not in IMAGE_EXTENSIONS:
                continue
            label = root / split / "labels" / (image.stem + ".txt")
            groups[group_key(image.stem)].append((image, label if label.exists() else None))
    return groups


def berry_count(items):
    total = 0
    for _, label in items:
        if label:
            total += sum(1 for line in label.read_text().splitlines()
                         if line.split() and int(line.split()[0]) == BERRY_CLASS)
    return total


def assign_folds(density, k=K, seed=SPLIT_SEED):
    rng = random.Random(seed)
    order = sorted(density, key=lambda g: (density[g], g))
    fold_of = {}
    for start in range(0, len(order), k):
        folds = list(range(k))
        rng.shuffle(folds)
        for group, fold in zip(order[start:start + k], folds):
            fold_of[group] = fold
    return fold_of


def fingerprint(fold_of):
    text = "\n".join(f"{g},{fold_of[g]}" for g in sorted(fold_of))
    return hashlib.sha256(text.encode()).hexdigest()[:12]


def role(fold, test_fold, k=K):
    if fold == test_fold:
        return "test"
    if fold in {(test_fold + 1) % k, (test_fold + 2) % k}:
        return "val"
    return "train"


def _link(src, dst):
    try:
        dst.symlink_to(src.resolve())
    except OSError:
        shutil.copy2(src, dst)


def split(args):
    root = Path(args.dataset)
    groups = collect_groups(root)
    density = {g: berry_count(items) for g, items in groups.items()}
    fold_of = assign_folds(density)
    digest = fingerprint(fold_of)
    print(f"images: {sum(len(v) for v in groups.values())}   groups: {len(groups)}   fingerprint: {digest}")
    if args.expected and digest != args.expected:
        raise SystemExit(f"fingerprint {digest} differs from {args.expected}: the dataset is not the same")

    out = Path(args.out) / f"fold{args.fold}"
    shutil.rmtree(out, ignore_errors=True)
    counts = defaultdict(lambda: [0, 0])
    for group, items in groups.items():
        part = role(fold_of[group], args.fold)
        for sub in ("images", "labels"):
            (out / part / sub).mkdir(parents=True, exist_ok=True)
        for image, label in items:
            _link(image, out / part / "images" / image.name)
            if label:
                _link(label, out / part / "labels" / label.name)
        counts[part][0] += len(items)
        counts[part][1] += density[group]

    names = yaml.safe_load((root / "data.yaml").read_text())["names"]
    data = dict(path=str(out.resolve()), train="train/images", val="val/images", test="test/images",
                nc=len(names), names=names)
    (out / "data.yaml").write_text(yaml.safe_dump(data, sort_keys=False, allow_unicode=True))
    (out / "split.json").write_text(json.dumps(
        dict(fold=args.fold, fingerprint=digest, images={p: c[0] for p, c in counts.items()},
             berries={p: c[1] for p, c in counts.items()}), indent=2))

    with open(Path(args.out) / "folds.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["group", "fold", "berries", "images"])
        for group in sorted(fold_of):
            writer.writerow([group, fold_of[group], density[group], ";".join(i.name for i, _ in groups[group])])

    for part in ("train", "val", "test"):
        print(f"{part:<6}{counts[part][0]:>5} images{counts[part][1]:>8} berries")

In [ ]:
FOLD = 0  # @param {type:"integer"}

split(SimpleNamespace(dataset="/content/dataset", fold=FOLD, out="/content/folds", expected="c6165896dbcf"))

## Training and test evaluation

The seed is fixed in every fold, so that the variation between folds comes
from the data partition. When the training ends, best.pt is evaluated on the
test fold.

In [ ]:
def evaluate(fold, run_dir, data, batch):
    """Metrics of best.pt on the test fold."""
    from ultralytics import YOLO

    model = YOLO(str(Path(run_dir) / "weights" / "best.pt"))
    results = model.val(data=data, split="test", imgsz=IMAGE_SIZE, batch=batch,
                        overlap_mask=False, max_det=MAX_DETECTIONS, plots=False,
                        project=str(run_dir), name="test", exist_ok=True)
    return metrics_dict(results, model.names)


def training_arguments(args, data):
    # The seed is the same in every fold, so that the variation between folds
    # comes from the data partition.
    return dict(
        data=str(data),
        batch=args.batch,
        seed=0,
        deterministic=True,
        device=args.device,
        workers=args.workers,
        project=args.project,
        name=f"fold{args.fold}",
        save_period=25,
        **SCHEDULE_VARIABILITY,
        **TRAINING,
        **ONLINE_AUGMENTATION,
    )


def train(args):
    from ultralytics import YOLO

    fold_dir = Path(args.folds) / f"fold{args.fold}"
    data = fold_dir / "data.yaml"
    if not data.exists():
        raise SystemExit(f"{data} not found; run the split step first")
    if args.resume:
        model = YOLO(str(Path(args.project) / f"fold{args.fold}" / "weights" / "last.pt"))
        results = model.train(resume=True)
    else:
        model = YOLO(args.model)
        results = model.train(**training_arguments(args, data))
    run_dir = Path(model.trainer.save_dir)
    write_record(args.fold, run_dir, fold_dir, args.project, args.batch, validation=metrics_dict(results, model.names))


def write_record(fold, run_dir, fold_dir, project, batch, validation=None):
    test = evaluate(fold, run_dir, str(fold_dir / "data.yaml"), batch)
    info = json.loads((fold_dir / "split.json").read_text())
    epoch, epochs_run = best_epoch(run_dir)
    record = dict(fold=fold, fingerprint=info["fingerprint"], images=info["images"],
                  best_epoch=epoch, epochs_run=epochs_run, test=test)
    if validation:
        record["validation"] = validation
    save_json(record, Path(project) / f"fold{fold}.json")
    print(table(test))

In [ ]:
BATCH = 8  # @param {type:"integer"}

args = SimpleNamespace(
    fold=FOLD, folds="/content/folds", project=f"{OUTPUT}/cv", batch=BATCH,
    model="yolo26l-seg.pt", device="0", workers=11, resume=False,
)
train(args)

## Resume an interrupted fold

Run the cells above again, except the training one, then this cell.

In [ ]:
args.resume = True
train(args)

## Summary

In [ ]:
def summary(args):
    records = {r["fold"]: r for r in (json.loads(p.read_text(encoding="utf-8"))
                                      for p in sorted(Path(args.results).glob("fold*.json")))}
    if not records:
        raise SystemExit(f"no fold<k>.json files in {args.results}")
    missing = sorted(set(range(K)) - set(records))
    if missing:
        print(f"missing folds: {missing}")
    fingerprints = {r["fingerprint"] for r in records.values()}
    if len(fingerprints) != 1:
        raise SystemExit(f"folds come from different splits: {fingerprints}")
    print(f"folds: {sorted(records)}   fingerprint: {fingerprints.pop()}")
    print(f"{'class':<18}{'type':<6}{'metric':<10}{'mean':>9}{'sd':>9}{'min':>9}{'max':>9}")
    for name in next(iter(records.values()))["test"]:
        for kind in ("box", "mask"):
            for metric in METRICS + ("F1",):
                values = [records[f]["test"][name][kind][metric] for f in sorted(records)]
                sd = statistics.stdev(values) if len(values) > 1 else float("nan")
                print(f"{name:<18}{kind:<6}{metric:<10}{statistics.mean(values):>9.4f}"
                      f"{sd:>9.4f}{min(values):>9.4f}{max(values):>9.4f}")

In [ ]:
summary(SimpleNamespace(results=f"{OUTPUT}/cv"))